In [ ]:
# =========================
# 1. IMPORTS + SEED
# =========================
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

# =========================
# 2. MAZE GRIDWORLD
# =========================
class GridWorld:
    def __init__(self, size=6, obstacles=None, sparse=False, shaped=False):
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.state = self.start
        self.sparse = sparse
        self.shaped = shaped
        self.obstacles = obstacles if obstacles else []

    def reset(self):
        self.state = self.start
        return np.array(self.state, dtype=np.float32)

    def step(self, action):
        x, y = self.state

        if action == 0: x -= 1
        elif action == 1: x += 1
        elif action == 2: y -= 1
        elif action == 3: y += 1

        x = max(0, min(self.size - 1, x))
        y = max(0, min(self.size - 1, y))

        new_state = (x, y)

        if new_state in self.obstacles:
            new_state = self.state

        self.state = new_state

        # Distance to goal
        dist = abs(self.goal[0] - x) + abs(self.goal[1] - y)

        if self.state == self.goal:
            return np.array(self.state, dtype=np.float32), 10, True

        # Reward shaping
        if self.shaped:
            reward = -dist * 0.1
        elif self.sparse:
            reward = 0
        else:
            reward = -1

        return np.array(self.state, dtype=np.float32), reward, False


# =========================
# 3. MAZE DESIGN (IMPORTANT)
# =========================
maze_obstacles = [
    (1,1), (1,2), (1,3),
    (2,3),
    (3,1), (3,2),
    (4,4),
    (2,1), (4,2)   # NEW
]

# =========================
# 4. DQN MODEL
# =========================
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        return self.net(x)
    
# =========================
# 5. TRAIN FUNCTION (IMPROVED)
# =========================
def train(env, model, episodes=400):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    rewards = []
    epsilon = 1.0

    for ep in range(episodes):
        state = torch.FloatTensor(env.reset())
        total_reward = 0
        done = False
        steps = 0

        while not done and steps < 50:
            # Epsilon decay
            if random.random() < epsilon:
                action = random.randint(0, 3)
            else:
                with torch.no_grad():
                    q_vals = model(state)
                    action = torch.argmax(q_vals).item()

            next_state, reward, done = env.step(action)
            next_state = torch.FloatTensor(next_state)

            with torch.no_grad():
                target = reward + 0.9 * torch.max(model(next_state))

            pred = model(state)[action]
            loss = loss_fn(pred, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            state = next_state
            total_reward += reward
            steps += 1

        # decay epsilon
        epsilon = max(0.1, epsilon * 0.995)

        rewards.append(total_reward)

    return rewards


# =========================
# 6. CURRICULUM STAGES
# =========================

# Stage 1 (Very Easy)
env1 = GridWorld(size=5, shaped=True)

# Stage 2 (Easy)
env2 = GridWorld(size=6, shaped=True)

# Stage 3 (Maze + shaped)
env3 = GridWorld(size=6, obstacles=maze_obstacles, shaped=True)

# Stage 4 (Maze + sparse = hardest)
env4 = GridWorld(size=6, obstacles=maze_obstacles, sparse=True)


# =========================
# 7. BASELINE TRAINING
# =========================
model_baseline = DQN()
baseline_rewards = train(env4, model_baseline, episodes=500)


# =========================
# 8. CURRICULUM TRAINING
# =========================
model_curriculum = DQN()

r1 = train(env1, model_curriculum, episodes=150)
r2 = train(env2, model_curriculum, episodes=150)
r3 = train(env3, model_curriculum, episodes=150)
r4 = train(env4, model_curriculum, episodes=150)

curriculum_rewards = r1 + r2 + r3 + r4


# =========================
# 9. SMOOTH GRAPH
# =========================
def smooth(data, window=15):
    return np.convolve(data, np.ones(window)/window, mode='valid')

